In [3]:
import os
import numpy as np
import tomopy as tp # standard reconstruction algorithms
import wget        # library for URL-downloads
import dxchange    # library fpr opening DX-files (tomographic format)

import matplotlib.pyplot as plt

# Image de la transformation de Radon

Dans cet exercice, vous êtes proposés deux tâches pratiques dont la résolution s'avère très avantageusement basée sur la structure de l'image de la transformation de Radon, qui est entièrement décrite par le théorème suivant :

**Théorème. (Gelfand-Graev-Helgason)** Soit $f\in \mathcal{S}(R^n)$. Alors $g(s,\theta) = Rf(s,\theta)$ satisfait aux propriétés suivantes :
1. $g\in\mathcal{S}(Z)$
2. $g(s,\theta) = g(-s, -\theta)$
3. pour tout $k = 0, 1, \dots$ les fonctions $I_k(\theta)$
\begin{equation}
    I_k(\theta) = \int\limits_{-\infty}^{+\infty}s^k Rf(s,\theta)\, ds, \, \theta \in S^{n-1}.
\end{equation}
sont les restrictions des polynômes homogènes de degré $k$ en $\theta_1, \theta_2, \dots, \theta_n$ à la sphère $S^{n-1}$. De plus, si pour la fonction $g(s,\theta)$ les conditions (1)-(3) sont satisfaites, alors il existe $f\in \mathcal{S}(R^n)$ telle que $g=Rf$.

## 1. Microtomographie (MicroCT) (0.5 point)

La microtomographie par rayons X (X-ray microtomography, MicroCT) est un autre domaine de la tomographie par rayons X qui se consacre à la reconstruction d'images de la structure d'objets petits (généralement jusqu'à quelques millimètres de diamètre).


<table>
<tr>
 <td> <img src="./microct-1.png" alt="micro-ct-1" style="width: 400px;"/> </td>
 <td> <img src="./microct-2.jpg" alt="micro-ct-2" style="width: 300px;"/> </td>
</tr>
<tr>
    <td> image d'une radioleaire obtenue par MicroCT</td>
    <td> section d'un os tibia de souris</td>
</tr>
</table>



Selon le modèle physique et l'utilisation des transformations de Radon, la microtomographie ne diffère pas de la tomographie par rayons X standard. Cependant, il y a quelques différences pratiques importantes :

  * en général - les cibles sont non vivantes (sauf si ce sont de petits animaux), ce qui signifie qu'il n'y a pas de restrictions sur la dose d'irradiation et donc pas de problèmes de bruit statistique
  * le faisceau de photons X est obtenu à l'aide d'un synchrotron, ce qui permet de garantir une divergence extrêmement faible du faisceau et même la cohérence optique des photons (conservation de la phase)
  * l'émetteur et le détecteur ne tournent pas autour de l'objet, mais l'objet est fixé sur un support et tourne autour de son axe ; grâce à cela, il n'y a pas de limitation sur le pas de discrétisation dans les données rayonnantes
  
Vous êtes invité à effectuer une reconstruction dans le cadre d'une microtomographie où l'objet irradié est une dent humaine.

**Exercice 1.1** En raison de cette résolution si fine, il y a un problème : le centre de rotation de l'objet n'est en réalité pas connu, et le définir manuellement avec une précision de l'ordre de la résolution est techniquement très difficile. En conséquence, les données rayonnantes sont déformées - le centre de rotation de l'objet ne correspond pas à la coordonnée 0 de la variable $s$ (voir figure). Vous devez corriger ces données et obtenir une image de section de la dent (où doivent apparaître les cavités et les fissures).

Le code ci-dessous charge les données rayonnantes et les convertit en format `numpy.ndarray`. Votre tâche consiste à reconstruire l'image de la dent à partir de ces données rayonnantes - elle doit montrer la structure interne de la dent et d'autres détails fins.

In [ ]:
# download h5 data with counts data

fname = 'tooth.h5'
data_status = os.path.exists('./' + fname)

if data_status == False:
    url = 'https://raw.github.com/fedor-goncharov/pdo-tomography-course/master/seminar-materials/seminar-3/tooth.h5'
    output_path = './'
    wget.download(url, output_path)

In [ ]:
# 1. preprocess data 
proj, flat, dark, theta = dxchange.read_aps_32id(
    fname = fname,
    sino = (0,2),
    )

# 1.1 normalize projection data and take -log
proj = tp.normalize(proj, flat, dark)
proj = tp.minus_log(proj)

# 1.2 convert to numpy.ndarray of size 181 x 640
proj = np.reshape(proj[:, 0, :], (181, 640))

# 1.3 (optional) plot image of raw data
plt.imshow(proj)
plt.title('MicroCT raw data')
plt.show()

In [8]:
# YOUR CODE HERE

## 2. Formules de Goncharov en microscopie électronique (0.5 point)

En microscopie électronique, les objets étudiés ont des dimensions nettement plus petites que dans le cas du MicroCT (de l'ordre de micromètres et moins). L'irradiation de l'objet est effectuée par un faisceau d'électrons, cependant, dans certaines modèles, les données initiales peuvent être modélisées par des transformations de Radon classiques.

<img src="./microscope.png" alt="micro-ct-1" style="width: 400px;"/>

Se pose ainsi le problème pratique suivant :
 * l'objet étudié ne peut pas être "tourné" en raison de sa petite taille (par exemple une bactérie ou une cellule). On peut considérer que l'objet se trouve dans un certain milieu et pivote aléatoirement, donc à différents moments, le microscope reçoit des données pour différentes directions $Rf(s,\theta_i), \, \theta_1, \theta_2, \dots, \theta_n$, mais les $\theta_i$ eux-mêmes sont **inconnus**.

Ainsi :
 1. À partir des données de rayonnement $Rf(s,\theta_i), \, \theta_1, \theta_2, \dots, \theta_n$, il faut reconstituer les directions $\theta_i$ (au moins avec une précision jusqu'à un pivot global).
 2. Reconstituer l'objet à partir des données de rayonnement $Rf(s,\theta_i), \, \theta_1, \theta_2, \dots, \theta_n$, en utilisant un algorithme approprié de reconstitution.

L'algorithme de reconstitution $\{\theta_i\}_{i=1}^n$ est le suivant :

1. $\theta_i = (\cos\varphi, \sin\varphi), \, \varphi \in [0, \pi)$, donc il suffit de reconstituer $\{\varphi_i\}_{i=1}^n$ avec une précision jusqu'à un pivot global. Définissons les moments 
\begin{equation}
    I_k(\varphi) = \int\limits_{-\infty}^{+\infty}s^k Rf(s,\theta(\varphi))\, ds, \, k = 0, 1, \dots
\end{equation}
2. Il n'est pas difficile de montrer que les moments $I_1(\varphi)$, $I_2(\varphi)$ peuvent être représentés sous la forme suivante (vous pouvez le démontrer si ce n'est pas évident)
\begin{equation}
    I_1(\varphi) = a \cos(\varphi - \alpha), \, I_2(\varphi) = b\cos(2(\varphi-\beta)) + c.
\end{equation}
Considérons la courbe $\Gamma$ dans le plan $R^2$ qui est définie de manière paramétrique comme suit :
\begin{align}
    \Gamma = \{x(\varphi), y(\varphi) | x(\varphi) = I_1(\varphi), \, y(\varphi) = I_2(\varphi), \, \varphi \in [0, 2\pi] \}
\end{align}

**Définition 1.** La courbe $\Gamma$ dans $R^2$ est appelée algébrique d'ordre $k$, si l'ensemble de ses points $(x,y)\in \Gamma$ peut être représenté comme la solution de l'équation 
\begin{equation}
    P(x,y) = 0,
\end{equation}
où 
\begin{equation}
    P \text{ - est un polynôme homogène de deux variables de degré }k. 
\end{equation}

**Lemme 1.** La courbe $\Gamma$ définie ci-dessus est une courbe algébrique de degré 4.

3. Un polynôme homogène de degré $4$ est défini par ses $15$ coefficients. Reconstituez la courbe $\Gamma$ (les coefficients du polynôme correspondant) à partir des données de rayonnement (même si nous ne connaissons pas les angles), en tenant compte du fait que les données de rayonnement contiennent 15 directions.

4. Pour reconstituer $a$, $b$, $c$, qui entrent dans la paramétrisation $I_1(\varphi)$, $I_2(\varphi)$, utilisons les formules suivantes (vérifiez-les)
\begin{equation}
    a = (x_{max} - x_{min})/2, \, b = (y_{max} - y_{min})/2, \, c = (y_{max} + y_{min})/2, 
\end{equation}
où $x_{max}, x_{min}$, $y_{min}$, $y_{max}$ sont des points appartenant à la courbe $\Gamma$.

5. En raison de la condition selon laquelle $\{\varphi_{i}\}_{i=1}^n$ doit être reconstitué jusqu'à un pivot global, nous considérerons que 
\begin{equation}
    \varphi_1 = 0.
\end{equation}
À partir de l'ensemble des données $\cos(\alpha), \, \cos(2\beta), \cos(\varphi_i-\alpha), \, \cos(2(\varphi_i - \beta))$, on peut reconstituer tous les angles $\varphi_i$, $i = 1, 2, \dots, 15$.

**Задача 2.1** Используйте алгоритм выше для того, чтобы найти направления 15-и неизвестных проекций. Код ниже загружаен лучевые данные с сервера.

**Problème 2.1** Utilisez l'algorithme ci-dessus pour trouver les directions des 15 projections inconnues. Le code suivant charge les données rayon depuis le serveur.

In [375]:
# download h5 data with counts data

fname = 'microscope-data.bin'
data_status = os.path.exists('./' + fname)

if data_status == False:
    url = 'https://raw.github.com/fedor-goncharov/pdo-tomography-course/master/seminar-materials/seminar-3/' + fname
    output_path = './'
    wget.download(url, output_path)

In [ ]:
nshift = 1024
ntheta = 150

projections = np.reshape(np.fromfile(fname, dtype=np.float), (ntheta, nshift))
plt.imshow(projections) # show unordered projection data

## Méthode itérative ART pour une géométrie de balayage arbitraire

Supposons que nous devons résoudre le système linéaire :
\begin{equation}
    Af = g, \, A\in \mathrm{Mat}(d, p), \, f\in R^p, \, g\in R^d.
\end{equation}

Supposons également que $f_k$ est l'approximation actuelle de $f$. Alors, on peut mettre à jour $f_k$ en projetant orthogonalement sur le plan affine

\begin{equation}
    a_i^Tf = g_i, \, i\in \{1, \dots, d\}.
\end{equation}

Notons par $P_i$ l'opérateur de projection orthogonale sur l'hyperplan $a_i^Tf = g_i$. Alors, l'algorithme ART peut être écrit comme suit :

```
    Algorithm ART
    
    set k=0, f(0) be the initial point
    repeat until convergence (set stopping criterion)
    
        u(0) := f(k)
        for n in (0, ..., d-1) do
            u(n+1) := Pn * u(n)
        endfor     
        f(k+1) := u(d)
        k := k + 1
        
    endrepeat
    
```

**Exercice 2.2** Utilisez l'algorithme ART (ou tout autre algorithme approprié pour résoudre ce système linéaire) afin de restaurer l'image originale à partir des données obtenues. *Indication :* pour faire fonctionner l'algorithme ART, vous devez être capable de calculer une ligne de la matrice de transformation de Radon - pour cela, vous pouvez utiliser le code de la première leçon ou le code fourni par l'enseignant `line_projector_siddon`, qui intègre exactement l'image le long d'une ligne. Vous devez également trouver la forme du projecteur $P_i$ par vous-même. Vous pouvez également utiliser la fonction standard `iradon_sart` de la bibliothèque `scikit.image` (voir la documentation).

**Paramètres :** l'image originale avait une taille de 128x128, 150 projections aléatoires, 1024 rayons dans chaque projection répartis uniformément sur l'intervalle [-1, 1].

In [ ]:
# YOUR CODE HERe